# Random Forest — nieuwe tuning binnen training

**Bijdrage en AI:** Codex maakte dit concept op verzoek van de gebruiker.
De mapnaam is geen bevestiging van Milans persoonlijke auteurschap of review.
De oorspronkelijke teamoplossing blijft behouden. De kritiek en lesverwijzingen
staan in [REVIEW_FINAL.md](REVIEW_FINAL.md).

Dit is ontwikkelonderzoek. De 900 al bekende ontwikkeltestrecords worden voor
dit experiment niet gescoord. De overblijvende projectdata zijn ook eerder
gebruikt; er ontstaat dus geen nieuwe onafhankelijke eindtest.

## De vooraf vastgelegde zoekruimte
We zoeken acht combinaties: max_features, min_samples_leaf en de categorische imputatie. De zoekkandidaat heeft 150 bomen om de rekentijd te begrenzen; de referentie behoudt de RF300-instellingen. Beide worden op dezelfde buitenfolds en in dezelfde omgeving opnieuw getraind. Dit is een vergelijking van complete procedures, geen geïsoleerd effect van één parameter.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'MushroomFinal/mushroom_project_dataset.csv').exists())
HERE = ROOT/'solutions/milan'
sys.path.insert(0, str(HERE))
from mushroom_alternative import development_data, pipeline, GRID, scores
X, y, ids, folds = development_data()
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import inspect
import mushroom_alternative as experiment
print(json.dumps(GRID, indent=2))
print(inspect.getsource(experiment.pipeline))
inner = StratifiedKFold(3, shuffle=True, random_state=20261016)
search_example = GridSearchCV(pipeline(), GRID, scoring='f1_macro', cv=inner, n_jobs=2, error_score='raise')
print(search_example)


{
  "classifier__max_features": [
    "sqrt",
    null
  ],
  "classifier__min_samples_leaf": [
    1,
    3
  ],
  "preprocessor__cat__impute__strategy": [
    "most_frequent",
    "constant"
  ]
}
def pipeline(trees=150):
    cats = [c for c in FEATURES if c not in NUMERIC]
    pre = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), NUMERIC),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent', fill_value='missing')),
            ('encode', OneHotEncoder(handle_unknown='ignore'))]), cats)])
    return Pipeline([('preprocessor', pre), ('classifier', RandomForestClassifier(
        n_estimators=trees, max_depth=20, min_samples_split=4,
        max_features=None, class_weight='balanced', random_state=42, n_jobs=1))])

GridSearchCV(cv=StratifiedKFold(n_splits=3, random_state=20261016, shuffle=True),
             error_score='raise',
             estimator=Pipeline(steps=[('preprocessor',
                                        Colu

## Waarom twee niveaus?
De binnenfolds kiezen instellingen met macro F1. Alleen buitenvalidatie beoordeelt die selectieprocedure. Ook de drempel voor 90% giftige recall wordt uit binnen-OOF voorspellingen gekozen en vervolgens buiten geëvalueerd. Dit is een strengere toepassing van de geleerde scheiding tussen training, validatie en test. Buitenfolds zijn hier ontwikkelvalidatie, geen nieuwe eindtest.

In [2]:
print(inspect.getsource(experiment.choose_threshold))
selections = json.loads((HERE/'results/selections.json').read_text(encoding='utf-8'))
assert len(selections) == 5
print(pd.json_normalize(selections).to_string(index=False))


def choose_threshold(y, probability):
    """Max specificity with >=90% recall on INNER OOF predictions only."""
    fpr, tpr, thresholds = roc_curve(y, probability, drop_intermediate=False)
    eligible = np.flatnonzero(np.isfinite(thresholds) & (tpr >= RECALL_TARGET))
    # Break equal-specificity ties by higher recall, then higher threshold.
    best = min(eligible, key=lambda i: (fpr[i], -tpr[i], -thresholds[i]))
    return float(thresholds[best])

 outer_fold  inner_cv_macro_f1  rf300_reference_threshold  rf300_reference_train_accuracy  rf_tuned_threshold  rf_tuned_train_accuracy best_parameters.classifier__max_features  best_parameters.classifier__min_samples_leaf best_parameters.preprocessor__cat__impute__strategy
          1           0.751418                   0.220207                        0.974085            0.219903                 0.971646                                     None                                             1                                       most_freq

## Werkelijk uitgevoerde grids
De tabellen zijn afkomstig van de afzonderlijk uitgevoerde trainingscode. De beste binnenscore is een selectiescore, geen onafhankelijke prestatiemeting.

In [3]:
for fold in range(1, 6):
    grid = pd.read_csv(HERE/f'results/inner_search_fold_{fold}.csv')
    assert len(grid) == 8
    print('Buitenfold:', fold)
    print(grid[['params','mean_train_score','mean_test_score','std_test_score','rank_test_score']].sort_values('rank_test_score').to_string(index=False))
run_receipt = json.loads((HERE/'results/run.json').read_text(encoding='utf-8'))
assert run_receipt['test_rows_scored'] == 0 and run_receipt['outer_evaluations'] == 20
print(json.dumps(run_receipt, indent=2))


Buitenfold: 1
                                                                                                                         params  mean_train_score  mean_test_score  std_test_score  rank_test_score
  {'classifier__max_features': None, 'classifier__min_samples_leaf': 1, 'preprocessor__cat__impute__strategy': 'most_frequent'}          0.981558         0.751418        0.001420                1
       {'classifier__max_features': None, 'classifier__min_samples_leaf': 3, 'preprocessor__cat__impute__strategy': 'constant'}          0.960350         0.751288        0.002776                2
       {'classifier__max_features': None, 'classifier__min_samples_leaf': 1, 'preprocessor__cat__impute__strategy': 'constant'}          0.991393         0.746259        0.010051                3
  {'classifier__max_features': None, 'classifier__min_samples_leaf': 3, 'preprocessor__cat__impute__strategy': 'most_frequent'}          0.947281         0.745183        0.001391                4
{'clas

## Optioneel de training opnieuw uitvoeren
De rapportcellen zijn al uitgevoerd en lezen bestaande nieuwe resultaten. Zet de vlag hieronder alleen op True als je de volledige search opnieuw wilt trainen. Dat schrijft uitsluitend resultaten in deze conceptmap en duurt enkele minuten.

In [4]:
RERUN_TRAINING = False
if RERUN_TRAINING:
    import joblib
    with joblib.parallel_config(backend='threading'):
        experiment.run()
else:
    print('Bestaande nieuwe training gecontroleerd; geen tweede training gestart.')


Bestaande nieuwe training gecontroleerd; geen tweede training gestart.
